# Phase 2 : Machine Learning Classique (Baseline)
Nous utilisons ici une Regression Logistique sur des vecteurs TF-IDF pour classer les avis en Positif ou Negatif.

In [1]:
import pandas as pd
import os
import sys
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_yelp_sample

In [2]:
# Preparation des donnees : on retire les notes neutres (3 etoiles)
df = load_yelp_sample('../data/raw/review.json', n_rows=50000)
df = df[df['stars'] != 3]

# Binarisation : 1 si note > 3 (Positif), sinon 0 (Negatif)
df['sentiment'] = df['stars'].apply(lambda x: 1 if x > 3 else 0)

# Separation train/test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['sentiment'], test_size=0.2)

Chargement de 50000 lignes depuis ../data/raw/review.json...


In [5]:
# Vectorisation du texte (TF-IDF) et Entrainement du modele
vec = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vec.fit_transform(X_train)

# On ajoute class_weight='balanced' pour pénaliser les erreurs sur la classe minoritaire
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_vec, y_train)

print(f'Precision sur le jeu d\'entrainement : {model.score(X_train_vec, y_train):.2%}')

Precision sur le jeu d'entrainement : 94.70%


In [6]:
# Evaluation sur le jeu de test
X_test_vec = vec.transform(X_test)
y_pred = model.predict(X_test_vec)

print('Rapport de classification :')
print(classification_report(y_test, y_pred))

# Sauvegarde du modele et du vectorizer pour usage futur
if not os.path.exists('../models'): os.makedirs('../models')
pickle.dump(model, open('../models/classic_model.pkl', 'wb'))
pickle.dump(vec, open('../models/vectorizer.pkl', 'wb'))

Rapport de classification :
              precision    recall  f1-score   support

           0       0.81      0.94      0.87      1819
           1       0.98      0.94      0.96      7046

    accuracy                           0.94      8865
   macro avg       0.89      0.94      0.91      8865
weighted avg       0.95      0.94      0.94      8865

